In [1]:
# selected_hive_numbers = [1,2,3,4,5,6]
selected_hive_numbers = [6]

from pipe import take
from pipe import select
from pipe import Pipe
import timestamps_and_frames
import features
import dataframes
import paths
import times
import  pandas
sink = lambda f: Pipe(lambda iterable: f(list(iterable)))
for hive_number in selected_hive_numbers: (
    hive_number
    | paths.hive_audio_folderpath_from_hive_number
    | timestamps_and_frames.hourly_audio_from_hive_folderpath(
        date_range=times.experiment_window
    )
    | features.hourly_features_from_audio_stream
    | dataframes.dataframe_from_hourly_stream(hive_number=hive_number)
    | select( lambda dataframe: 
        dataframes.saved_csv_filepath_from_features_dataframe(
            dataframe=dataframe,
            dataframe_filename=paths.audio_features_filename_from_hive_number(hive_number),
        )
    )
    # | take(2)
    | select(lambda filepath:
        pandas.read_csv(filepath, parse_dates=["timestamp"]).head()
    )
    | sink(list)
)

proessing ../../data/audio/hive_06/03/07/hive_06_20260307_170247.flac
proessing ../../data/audio/hive_06/03/07/hive_06_20260307_173247.flac
	hour_timestamp=np.datetime64('2026-03-07T17:00:00.000000')
proessing ../../data/audio/hive_06/03/07/hive_06_20260307_180247.flac
proessing ../../data/audio/hive_06/03/07/hive_06_20260307_183247.flac
	hour_timestamp=np.datetime64('2026-03-07T18:00:00.000000')
proessing ../../data/audio/hive_06/03/07/hive_06_20260307_190247.flac
proessing ../../data/audio/hive_06/03/07/hive_06_20260307_193310.flac
	hour_timestamp=np.datetime64('2026-03-07T19:00:00.000000')
proessing ../../data/audio/hive_06/03/07/hive_06_20260307_200310.flac
proessing ../../data/audio/hive_06/03/07/hive_06_20260307_203310.flac
	hour_timestamp=np.datetime64('2026-03-07T20:00:00.000000')
proessing ../../data/audio/hive_06/03/07/hive_06_20260307_210310.flac
proessing ../../data/audio/hive_06/03/07/hive_06_20260307_213310.flac
	hour_timestamp=np.datetime64('2026-03-07T21:00:00.000000')


In [2]:
raise

RuntimeError: No active exception to reraise

In [ ]:
from pipe import select
from pipe import Pipe
import timestamps_and_frames
import features
import dataframes
import paths
import times
sink = lambda f: Pipe(lambda iterable: f(list(iterable)))
for hive_number in selected_hive_numbers: (
    hive_number
    | paths.hive_audio_folderpath_from_hive_number
    | timestamps_and_frames.hourly_audio_from_hive_folderpath
    | features.hourly_features_from_audio_stream
    | dataframes.dataframe_from_hourly_stream(hive_number=hive_number)
    | select( lambda dataframe: 
        dataframes.saved_csv_filepath_from_features_dataframe(
            dataframe=dataframe,
            dataframe_filename=paths.audio_features_filename_from_hive_number(hive_number),
        )
    )
    | select(lambda filepath:
        pandas.read_csv(filepath, parse_dates=["timestamp"]).head()
    )
    | sink(list)
)

In [ ]:
raise

In [ ]:
from pipe import take

import timestamps_and_frames
import features
import dataframes
import paths
import times

test_dataframe = (
    3
    | paths.hive_audio_folderpath_from_hive_number
    | timestamps_and_frames.hourly_audio_from_hive_folderpath
    # | take(10)
    | features.hourly_features_from_audio_stream
    | dataframes.dataframe_from_hourly_stream(hive_name="hive_03")
)
print(test_dataframe.shape)
print(test_dataframe.columns.tolist())
test_dataframe

In [ ]:
import matplotlib.pyplot
import matplotlib.dates

plot_features = [
    "root_mean_square_energy_mean",
    "spectral_centroid_mean",
    "spectral_flatness_mean",
    "spectral_flux_mean",
    "high_root_mean_square_energy_mean",
    "low_to_high_energy_ratio_mean",
    "low_modulation_energy",
    "zero_crossing_rate_mean",
]

existing = [f for f in plot_features if f in test_dataframe.columns]
dates = matplotlib.dates.date2num(test_dataframe["timestamp"])

figure, axes = matplotlib.pyplot.subplots(
    len(existing), 1,
    figsize=(18, 3 * len(existing)),
    sharex=True,
)

for axis, name in zip(axes, existing):
    axis.plot_date(dates, test_dataframe[name].values, fmt="o-", markersize=3, linewidth=0.8)
    axis.set_ylabel(name, fontsize=7)
    axis.xaxis.set_major_locator(matplotlib.dates.DayLocator())
    axis.xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%m/%d"))
    axis.xaxis.set_minor_locator(matplotlib.dates.HourLocator(interval=6))
    axis.xaxis.set_minor_formatter(matplotlib.dates.DateFormatter("%H:%M"))
    axis.tick_params(axis="x", which="minor", labelsize=5, rotation=45)
    axis.tick_params(axis="x", which="major", labelsize=7, rotation=45)
    axis.grid(which="major", alpha=0.3)
    axis.grid(which="minor", alpha=0.1)

figure.suptitle("hive 03 — hourly features")
figure.tight_layout()
matplotlib.pyplot.show()

In [ ]:
raise

In [ ]:
from pipe import select, chain, where, take, tee, Pipe, sort

stow = lambda f: Pipe(lambda args, **kwargs: f(*args, **kwargs))
compose = lambda f, g: (lambda *args, **kwargs: g(*f(*args, **kwargs)))
sink = lambda f: Pipe(lambda iterable: f(list(iterable)))
def identity(x): return x